In [ ]:
from markdown_it.common.entities import entities
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from openai import OpenAI
from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import load_gliner_model, predict_dataset
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.llm import format_prompt_for_ner, predict_llm_entities

In [ ]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_clean'
OUTPUT_PATH = '../results/llm/predictions_llm.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:
gold_samples = load_jsonl(VALIDATION_PATH)
responses = []

for sample in gold_samples:
    text = sample.get("text", "")
    doc_id = sample.get("id", "")

    entities = predict_llm_entities(text, LABELS, client= client, model="gpt-4o-mini")

    responses.append({
            "id": doc_id,
            "text": text,
            "entities": entities
        })

pdf_samples = extract_pdf_dataset(PDF_DIR, gold_samples)
responses = predict_dataset(model, pdf_samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)

In [ ]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)